# White Castle rules assistant

An assistant that answers questions about the rules of the board game **White Castle**, in Spanish. It uses RAG (retrieval-augmented generation): instead of answering from memory, it first looks up the relevant passages of the official rulebook and then answers from them, citing the page. It remembers each conversation, so it understands follow-up questions.

**The starting point** is the Spanish rulebook: a 20-page scanned PDF where every page is an image, with no text to search. The notebook follows the path that turns it into an assistant:

| Section | Step | What happens |
|---|---|---|
| 1.1 | PDF → text | Gemini transcribes each page image to Markdown |
| 1.2 | Text → chunks | The text is cleaned and split into chunks that keep their page and section |
| 1.3 | Chunks → vectors | Gemini embeddings, stored in a local ChromaDB collection |
| 1.4 | Question → rules | Test questions check that the search finds the right rules |
| 2 | Rules → answer | LangGraph agent with memory: it rewrites follow-up questions, retrieves the rules and answers with Gemini |
| 3 | Conversation | Documented example questions and a chat cell |

**Why each decision:** every section explains the reasons behind its choices. Many of them come from tests run while building the agent. Those tests are not kept in the notebook, to keep it short, but what they showed is told where each decision is explained.

How to install and run it (dependencies, Gemini API key, where to place the rulebook): see the README.

## 0. Setup

Imports, the configuration shared by the whole notebook (paths and models), the Gemini API key, read from `.env` and never printed, and the chat model, shared by the transcription (section 1.1) and the agent (section 2). The parameters of each step live in its own section, next to the explanation of their values.
- **Why `gemini-3.5-flash-lite`:** it is free, and its daily quota is enough for a full run and a live demo. `gemini-3.8-flash` was tried on the questions Flash-Lite got wrong, but its free tier allows only 20 requests per day, each turn of a conversation spends two (rewrite and answer), and its first calls failed with `503` (Google's servers busy): it cannot hold a live demo. The one answer it gave was right, but Flash-Lite gets that question right too in most runs.
- **Spaced calls:** the free tier of Flash-Lite allows 15 requests per minute, and the transcription alone makes 20 in a row. Without a limit the run stopped with a `429` error, so the chat model spaces its calls to 12 per minute (`CHAT_REQUESTS_PER_SECOND`).

In [1]:
# Imports
import base64
import os
import re
import statistics
from pathlib import Path
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pypdf import PdfReader

In [2]:
# Paths
PROJECT_DIR = Path.cwd()
ENV_PATH = PROJECT_DIR / ".env"
PDF_PATH = PROJECT_DIR / "data" / "white_castle.pdf"
TRANSCRIPT_DIR = PROJECT_DIR / "data" / "transcript"
CHROMA_DIR = PROJECT_DIR / "chroma_db"

# Models
CHAT_MODEL = "gemini-3.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-2"
CHAT_REQUESTS_PER_SECOND = 0.2  # 12 per minute: the free tier allows 15

In [3]:
def load_api_key():
    """Gemini API key: .env → environment variable GOOGLE_API_KEY."""
    load_dotenv(ENV_PATH)
    if not os.getenv("GOOGLE_API_KEY"):
        raise ValueError(f"GOOGLE_API_KEY is empty or missing in {ENV_PATH}")


load_api_key()
llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, rate_limiter=InMemoryRateLimiter(requests_per_second=CHAT_REQUESTS_PER_SECOND))

## 1. Knowledge base

The agent can only answer as well as it searches. This part turns the scanned rulebook into a knowledge base the agent can search, and ends by checking that the search finds the right rules.

### 1.1 From a scanned PDF to text

The PDF has no text layer: each page is a single JPEG image. Instead of OCR, Gemini reads each image and transcribes it to Markdown:
- **One page per request:** with the whole PDF at once the model tends to summarise, and the page of each rule would be lost. If a page fails, only that page is repeated.
- **Icons that carry rules are described:** in a board game many rules (costs, rewards) only appear as icons, so they are described briefly between brackets, for example `[Icono: 2 monedas]`. Decorative illustrations are ignored.
- **Done once:** each page is saved to `data/transcript/page_NN.md` and skipped in later runs, so the quota is only spent once.
- **Spaced calls:** the 20 pages are more requests than the free tier allows per minute, so the chat model of section 0 spaces them.

In [4]:
# Constants
TRANSCRIPTION_PROMPT = """You are transcribing one page of the Spanish rulebook of the board game White Castle.
- Transcribe all the text literally, in Spanish. Do not translate, summarize or correct it.
- Mark section titles with Markdown headings: # for main titles, ## for subsections.
- When an icon, diagram or illustrated example carries rules information, describe it briefly in Spanish between square brackets, for example: [Icono: 2 monedas].
- Ignore purely decorative illustrations, page numbers and repeated headers or footers.
- Return only the transcription, with no comments before or after it."""

In [5]:
def transcribe_manual():
    """Rulebook PDF → one Markdown file per page in TRANSCRIPT_DIR (pages already done are skipped)."""
    TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
    pages = PdfReader(PDF_PATH).pages
    transcribed = [transcribe_page(page, number) for number, page in enumerate(pages, start=1)]
    print(f"{len(pages)} pages: {sum(transcribed)} transcribed now, {len(pages) - sum(transcribed)} already on disk")


def transcribe_page(page, number):
    """One page: scanned image → Gemini transcription → page_NN.md (False if it was already on disk)."""
    output_path = TRANSCRIPT_DIR / f"page_{number:02d}.md"
    if output_path.exists():
        return False
    message = build_page_message(page)
    reply = llm.invoke([message])
    output_path.write_text(reply.text, encoding="utf-8")
    print(f"Page {number:02d}: transcribed ({len(reply.text)} characters)")
    return True


def build_page_message(page):
    """Message for Gemini: transcription prompt + the page JPEG image as base64."""
    image_b64 = base64.b64encode(page.images[0].data).decode()
    return HumanMessage(content=[
        {"type": "text", "text": TRANSCRIPTION_PROMPT},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"},
    ])


transcribe_manual()

20 pages: 0 transcribed now, 20 already on disk


### 1.2 Cleaning and chunking

The agent does not search whole pages but chunks: pieces small enough to hold one rule each. The transcription becomes chunks in four steps:
- **Cleaning:** removes the transcription noise that repeats (formula marks, arrow bullets, warning signs, decorative images). Isolated typos are left as they are: fixing them by hand does not pay off for the search.
- **Sections first:** the text is split by its Markdown headings, so a rule is not cut in two. A page that starts mid-section keeps the section of the previous page.
- **Then size:** sections longer than `CHUNK_SIZE` characters are split again, with some overlap so that a sentence on the border is not lost.
- **Each chunk carries its context:** its page and section are saved as metadata, to cite the source. The section path is also written at the start of the text, because the embedding only sees the text: a loose sentence like "pay 2 coins" means little without its section.

In [6]:
# Constants
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3")]

# Cleaning rules: (regex pattern, replacement)
CLEANING_RULES = [
    (r"\$([^$\n]+)\$",                        r"\1"),  # formula marks: $10+10$ → 10+10
    (r"\[Icono: Flecha roja\]\s*",            "- "),   # arrow bullets → list items
    (r"\[Icono: Signo de exclamación\]\s*",   ""),     # warning sign before "Atención"
    (r"\*?\[Ilustración decorativa:[^\]]*\]\*?", ""),  # decorative images
    (r"\n{3,}",                               "\n\n"), # extra blank lines
]

In [7]:
def build_chunks():
    """Transcript pages → cleaned text → sections → chunks with page and section metadata."""
    pages = load_pages()
    sections = split_into_sections(pages)
    chunks = split_long_sections(sections)
    print_chunk_summary(chunks)
    return chunks


def load_pages():
    """Transcript files → list of (page number, cleaned text)."""
    pages = []
    for path in sorted(TRANSCRIPT_DIR.glob("page_*.md")):
        number = int(path.stem.split("_")[1])
        pages.append((number, clean_text(path.read_text(encoding="utf-8"))))
    return pages


def clean_text(text):
    """Raw transcription → text without the noise listed in CLEANING_RULES."""
    for pattern, replacement in CLEANING_RULES:
        text = re.sub(pattern, replacement, text)
    return text.strip()


def split_into_sections(pages):
    """
    Pages → one document per section:
        - Metadata: headings (h1, h2, h3) and page number
        - Text at the top of a page with no heading keeps the section of the previous page
    """
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS)
    sections = []
    last_headers = {}
    for number, text in pages:
        for section in splitter.split_text(text):
            if section.metadata:
                last_headers = dict(section.metadata)
            else:
                section.metadata = dict(last_headers)
            section.metadata["section"] = " > ".join(section.metadata.values())
            section.metadata["page"] = number
            sections.append(section)
    return sections


def split_long_sections(sections):
    """Sections → chunks of at most CHUNK_SIZE characters, each starting with its section title."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(sections)
    for chunk in chunks:
        chunk.page_content = f"{chunk.metadata['section']}\n\n{chunk.page_content}"
    return chunks


def print_chunk_summary(chunks):
    """Chunk count, length statistics and one example chunk."""
    lengths = [len(chunk.page_content) for chunk in chunks]
    print(f"Chunks: {len(chunks)}")
    print(f"Length (characters): min {min(lengths)}, median {statistics.median(lengths):.0f}, max {max(lengths)}")
    print(f"\nExample (chunk 10, page {chunks[10].metadata['page']}):\n{chunks[10].page_content}")


chunks = build_chunks()

Chunks: 76
Length (characters): min 118, median 616, max 1087

Example (chunk 10, page 2):
PREPARACIÓN DE LA PARTIDA > PUENTES

7 [Icono: 7 en cuadro rojo] Se colocan los puentes en sus espacios del río (si lo preferís, podéis jugar sin los puentes troquelados). Tomad 3/4/5 dados de cada color en función de si seis 2/3/4 jugadores. Devolved a la caja los dados que sobren.  
Tirad los dados de cada color y colocadlos en el puente que les corresponda, ordenándolos de manera ascendente según su valor de izquierda a derecha. Es decir, el valor más bajo a la izquierda del puente junto a la linterna, y el más alto a la derecha. Los demás se colocan en el centro del puente, en orden.


### 1.3 Vector store

Each chunk is turned into a vector with Gemini embeddings: chunks that talk about similar things get close vectors, so a question can find its rules by meaning and not only by exact words (in the tests, "farol" found the rule of the bridge's "linterna"). The vectors are stored in a local ChromaDB collection (`chroma_db/`):
- If the collection already exists, it is loaded from disk and no embeddings are spent
- After changing the cleaning or chunking settings, set `REBUILD_VECTOR_STORE = True` once to rebuild it: the collection cannot tell by itself that the chunks have changed

In [8]:
# Constants
COLLECTION_NAME = "white_castle_rules"
REBUILD_VECTOR_STORE = False

In [9]:
def load_vector_store(chunks):
    """Chunks → ChromaDB collection in CHROMA_DIR (loaded from disk if it is already built)."""
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL),
        persist_directory=str(CHROMA_DIR),
    )
    stored = len(vector_store.get()["ids"])
    if REBUILD_VECTOR_STORE or stored == 0:
        index_chunks(vector_store, chunks)
    else:
        print(f"Vector store loaded from disk: {stored} chunks")
    return vector_store


def index_chunks(vector_store, chunks):
    """Empty the collection, then embed and store every chunk with a stable id."""
    vector_store.reset_collection()
    ids = [f"chunk_{number:03d}" for number in range(len(chunks))]
    vector_store.add_documents(chunks, ids=ids)
    print(f"Vector store built: {len(chunks)} chunks indexed")


vector_store = load_vector_store(chunks)

Vector store loaded from disk: 76 chunks


### 1.4 Does the search find the right rules?

Before building the agent on top of it, the search is checked with questions whose answer is known to be on a given page of the rulebook: direct questions about key rules, and harder ones written as a player would ask them, which need several rules at once.

For each question, the table shows the expected pages, the positions among the top `RETRIEVAL_K` chunks where a chunk from those pages appears, and the top chunk.

In [10]:
# Constants
RETRIEVAL_K = 5

LONG_DICE_QUESTION = (
    "En The White Castle, un jugador quiere coger un dado de valor 2 y colocarlo en un espacio cuyo valor de referencia es 5. "
    "Tiene 1 moneda, por lo que no puede pagar las 3 monedas de diferencia. Sin embargo, al coger ese dado "
    "activa su bonificación de linterna, que le proporcionaría monedas suficientes para cubrir la diferencia. "
    "¿Puede realizar la jugada? Explica el orden exacto de resolución, si puede utilizar la recompensa de la "
    "linterna para pagar el dado y qué alternativa tiene si no puede efectuar esa colocación."
)

LONG_GARDENER_QUESTION = (
    "Al final de la primera ronda, un jugador activa un jardinero que ya tenía colocado. La acción de ese jardinero "
    "le permite colocar un segundo jardinero en otro jardín bajo un puente que contiene dados. El jugador sostiene que, "
    "como el segundo jardinero también está bajo un puente con dados, debe activar inmediatamente su acción de final de "
    "ronda. Además, afirma que no debe pagar de nuevo el coste de arroz de ese segundo jardinero, pero que tampoco "
    "necesita pagar las monedas exigidas por la acción que decida ejecutar. Evalúa por separado las tres afirmaciones. "
    "Después, explica qué sucede si la misma situación se produce al final de la tercera ronda."
)

# Direct questions: (question, rulebook pages with the answer)
DIRECT_QUESTIONS = [
    ("¿Se pueden apilar dados en una partida a 2 jugadores?",              (5,)),
    ("¿Qué pasa si coloco un dado de valor inferior al de la casilla?",     (5,)),
    ("¿Cuántos cortesanos caben en la puerta del castillo?",                (10,)),
    ("¿Cuál es el máximo de recursos de cada tipo que puedo tener?",        (5,)),
    ("¿Cuánto cuesta subir un cortesano dos pisos?",                        (10,)),
    ("¿Cómo puntúan los guerreros al final de la partida?",                 (14,)),
    ("En el modo solitario, ¿el clan rival puntúa los recursos sobrantes?", (18,)),
]

# Harder questions, as a player would ask them: (question, rulebook pages with the answer)
HARD_QUESTIONS = [
    ("¿Puedes pagar un dado con los recursos que obtienes de la propia acción?",            (5, 8)),
    ("¿Puedes ascender dos veces o colocar dos cortesanos en una misma acción?",            (10,)),
    ("¿Puede un jardinero recién colocado activarse inmediatamente al final de una ronda?", (13, 10)),
    (LONG_DICE_QUESTION,                                                                    (5, 8)),   # full game situation, "linterna" for "lámpara"
    (LONG_GARDENER_QUESTION,                                                                (13, 10)), # full game situation, three claims to check
]

TEST_QUESTIONS = DIRECT_QUESTIONS + HARD_QUESTIONS

In [11]:
def check_retrieval():
    """Every test question → one table row, then how many questions found an expected page."""
    print(f"{'Question':<58} {'Expected':<9} {'Found at':<10} Top chunk")
    found = 0
    for question, expected_pages in TEST_QUESTIONS:
        chunks = vector_store.similarity_search(question, k=RETRIEVAL_K)
        ranks = [rank for rank, chunk in enumerate(chunks, start=1) if chunk.metadata["page"] in expected_pages]
        found += bool(ranks)
        print_row(question, expected_pages, ranks, chunks[0])
    print(f"\n{found}/{len(TEST_QUESTIONS)} questions with a chunk from an expected page in the top {RETRIEVAL_K}")


def print_row(question, expected_pages, ranks, top_chunk):
    """
    One aligned table row:
        Example: '¿Cuánto cuesta subir...'  p10  1, 3, 4  p10 CORTESANOS
        - Questions longer than the column are shortened
        - Icon descriptions are removed from the section title
    """
    short_question = question if len(question) <= 56 else question[:53] + "..."
    expected = ", ".join(f"p{page}" for page in expected_pages)
    found_at = ", ".join(str(rank) for rank in ranks) or "-"
    section = re.sub(r"\s*\[Icono:[^\]]*\]", "", top_chunk.metadata["section"])
    top = f"p{top_chunk.metadata['page']} {section[:45]}"
    print(f"{short_question:<58} {expected:<9} {found_at:<10} {top}")


check_retrieval()

Question                                                   Expected  Found at   Top chunk


¿Se pueden apilar dados en una partida a 2 jugadores?      p5        1, 3       p5 APILAR DADOS:


¿Qué pasa si coloco un dado de valor inferior al de l...   p5        1, 3       p5 USO DE LOS DADOS > PROCEDIMIENTO PARA USAR LO


¿Cuántos cortesanos caben en la puerta del castillo?       p10       1, 2, 3, 4 p10 CORTESANOS > A


¿Cuál es el máximo de recursos de cada tipo que puedo...   p5        1          p5 OBTENER Y PAGAR RECURSOS:


...

¿Puedes pagar un dado con los recursos que obtienes d...   p5, p8    1, 2, 3, 4 p8 EL DOMINIO PERSONAL


¿Puedes ascender dos veces o colocar dos cortesanos e...   p10       1, 2, 3, 4 p10 CORTESANOS


¿Puede un jardinero recién colocado activarse inmedia...   p13, p10  1, 2, 3, 5 p13 FINAL DE LA RONDA


En The White Castle, un jugador quiere coger un dado ...   p5, p8    3, 5       p6 EL CASTILLO


Al final de la primera ronda, un jugador activa un ja...   p13, p10  1, 2, 3, 5 p13 FINAL DE LA RONDA

12/12 questions with a chunk from an expected page in the top 5


**What the check shows**
- **Direct questions:** the top chunk comes from the expected page in all seven.
- **Harder questions, short form:** the top chunk also comes from an expected page. When the answer is spread over two pages (paying for a die: pages 5 and 8; gardeners: pages 13 and 10), both pages appear in the top 5.
- **Long questions:** the gardener one finds both pages in the first two positions. The dice one is the weakest case: the top chunk is *El castillo* (page 6), pulled by the words "The White Castle" of the question, and the expected chunks appear in positions 3 (paying the difference) and 5 (the order of the steps).
- **Why `RETRIEVAL_K = 5`:** with 3 chunks, the long dice question would lose the order of the steps, which is half of its answer. Five chunks of at most about 1,100 characters are still a small context for Gemini.
- **The search brings the pieces, not the answer:** in the harder questions the rulebook never states the answer literally (for example, "the lamp reward cannot pay for the die"). It has to be deduced from the order of the steps, which is the job of the agent and its system prompt (section 2).

## 2. RAG agent with memory

The agent is a LangGraph graph: a small pipeline where a shared **state** travels through **nodes** (plain Python functions) following the **edges** between them. Each node reads the state and adds its result to it.

```
START → rewrite → retrieve → generate → END
```

- **rewrite:** earlier turns + last question → kind of question and a standalone question to search (section 2.2)
- **retrieve:** the standalone question → the top `RETRIEVAL_K` chunks of the rulebook (section 1)
- **generate:** system prompt + the whole conversation + the chunks → Gemini's answer (section 2.1)

The core of RAG is `retrieve → generate`. Memory adds `rewrite` in front and a checkpointer that saves the state of the graph after every turn (section 2.2). That is why it is a graph and not two function calls in a row: the graph itself keeps each conversation, and each new step is just a new node.

### 2.1 System prompt

The system prompt sets the role of the agent, the format of its answers and its limits. Each rule has a reason, and most of them fix a failure seen in the tests:

| Rule | Why |
|---|---|
| Expert, short answers | Players ask in the middle of a game: they need the answer, not an essay |
| Spanish answers, English prompt | The rulebook and the players are Spanish; models follow instructions in English best |
| Only the excerpts | Without it, the model fills the gaps with what it believes about the game, which may be wrong |
| Kind of question first | Each kind of question has its own format. With all the rules in one list, the model added a page even to questions it could not answer |
| Short answer → explanation → page | The first sentence settles the doubt at the table, the paragraph explains why, and the page lets the player check it |
| "Sí" or "No" only for yes/no questions, parts not numbered | With the answer format as a numbered list, the model started open questions with "Sí", and in the "learning to play" mode it numbered the three parts instead of the steps. The question words ("qué", "cuánto"...) are named because a general rule was not enough |
| Limits read literally | In the tests, the model misread the limit of the courtier action ("las dos acciones una vez cada una": each one at most once) about half of the time |
| Deduce | Tricky answers are rarely written literally in the rulebook (see 1.4) |
| Ask when the question cannot be understood on its own | Without it, a follow-up with no conversation behind it ("¿Y luego cómo lo subo?") made the model guess what it was about. Only in the first turn: otherwise it also asked when the earlier turns made the question clear |
| "No encuentro esta regla en el manual" + house rule | With only a few chunks, the agent cannot tell a gap in the rulebook from a failed search: the sentence is honest about both. The house rule keeps the game going, labelled as unofficial |
| One step at a time to learn the game | A whole explanation of the game is a wall of text. It needs the memory of section 2.2. The first answer is only the list: otherwise the model explained the first step without waiting |
| Fixed reply outside White Castle | Keeps the agent in its role. Worded freely, the model still added a page |

In [12]:
# Constants
SYSTEM_PROMPT = """You are an expert on the rules of the board game White Castle. You help players during a game, so your answers are short and clear. Always answer in Spanish.

You answer only from the rulebook excerpts given with each question. Each excerpt shows its page and section. Never invent a rule or answer from general knowledge of the game.

First decide which of these five kinds of question it is, and answer only as its section says.

# 1. Rules question answered by the excerpts
Answer in three parts, written as plain text, without numbering or labelling them:
- Short answer, in the first sentence. Start with "Sí" or "No" only when the question asks whether something can be done or is true ("¿Puedo...?", "¿Se puede...?", "¿Es...?"). Questions with "qué", "cuánto", "cuándo", "cómo" or "cuál" never start with "Sí" or "No": start with the answer itself (a number, a cost, a moment of the game...).
- Explanation: one short paragraph with the details and the reasoning, so the player understands why.
- Source: the pages of the excerpts you used, for example "Lo tienes en la página 5."

Read quantities and limits literally (once, each, at most, only one): they are often the key to the answer. If the answer is not written literally but follows from the excerpts (for example, from the order of the steps), deduce it and explain the reasoning in the paragraph.

# 2. Question that cannot be understood on its own
Only for the first question of a conversation, when there are no earlier turns: it refers to something that was never said ("eso", "lo", "y luego", "en ese caso"). Do not guess what it means: ask the player, in one or two sentences, which rule or part of the game they mean. No page.
If there are earlier turns, the references point to them: never ask, answer with the kind that fits.

# 3. Rules question the excerpts do not answer
1. Start with "No encuentro esta regla en el manual" and say briefly that the rulebook may leave it open, or the search may not have found it. Never present an assumption as an official rule.
2. Propose a house rule for this game, clearly labelled as a suggestion, not an official rule.
Do not add a "Lo tienes en la página..." line: no page has the rule.

# 4. Learning to play
For broad questions, such as "enséñame a jugar", "cómo se prepara la partida" or "cómo se juega":
1. List the main steps as a short numbered stepper, one line per step.
2. Ask if the player is ready for the first step. This first answer has only the list and the question: do not explain any step yet.
3. Explain one step at a time: a short paragraph with its page, not the three parts of kind 1. Move to the next step only when the player confirms.
Never explain all the steps at once.

# 5. Not about White Castle
Reply only with "Solo puedo ayudarte con las reglas de White Castle.", with no explanation and no page."""

### 2.2 Memory

Memory needs two pieces:
- **Keeping the conversation:** a **checkpointer** saves the state of the graph after every turn, under the id of its conversation (`thread_id`), and loads it back at the start of the next turn. Like a waiter's notepad with one sheet per table: each table only sees its own orders. `InMemorySaver` keeps the conversations in memory, so they are lost when the kernel restarts: the demo and the review run in a single session, and saving them to disk would add a dependency for nothing.
- **Searching with follow-up questions:** "¿Y si somos dos?" means nothing to the vector store on its own. The `rewrite` node first turns it into a question that can be understood without the conversation.

From the second turn on, Gemini sorts the last question into one of two kinds:

| Kind | Example | Search with | Note to `generate` |
|---|---|---|---|
| `same_topic` | "¿Y luego cómo lo subo?" | the question rewritten to stand on its own (unchanged if it already does) | what the question means on its own, if it was rewritten |
| `new_topic` | a clear question about another topic | the question as it is | "the player has changed topic" |

The rewrite was tuned with five test conversations of up to five turns: clear questions about one topic, follow-ups, unannounced changes of topic, a follow-up in a new conversation and the "learning to play" mode. Each choice below comes from them:
- **Why the kind and not only the rewrite:** a first version only rewrote, and it tended to carry the earlier topic into clear questions (a question about stacking dice was searched "on the bridges" because the bridges had come up before). Naming the case of a new topic lets the rewrite leave it clean.
- **Why two kinds and not three:** a version with a third kind, clear questions about the same topic searched as written, sorted "¿Puedo hacer las dos cosas en el mismo turno?" there, so "las dos cosas" was searched as it was and the wrong rules came back. With two kinds it was searched as "¿Puedo realizar la acción de solicitar audiencia y la acción de ascensión social en el mismo turno?", and the courtier rule came first. A doubtful question is at worst rewritten when it did not need it, which does no harm.
- **Same or new topic is not reliable, but harmless:** clear questions about the same topic were often classified `new_topic`. Both kinds search the question as written, and the note only asks `generate` to answer it on its own.
- **The rulebook's words:** the rewrite uses the words the agent used in its answers, which come from the rulebook, instead of the player's informal ones ("la sala del Daimio", not "arriba del todo"). Points are named "puntos de clan", as the rulebook always calls them: asked "¿Y cuántos puntos da?" after the cost of moving a courtier up, the search with "puntos" brought the courtier action again and missed the scoring of page 14. The same goes for the steps of the "learning to play" mode: named by their content, not by their number ("the second step"), or the search misses them.
- **The topic-change note goes to `generate`, not to the search:** words from the earlier topic in the searched question would pull chunks of that topic. `generate` already sees the earlier turns, so the note only tells it not to mix them into the answer.
- **`generate` answers the player's own words, with the rewrite as a note:** the rewrite can lose a detail, and `generate` sees the whole conversation anyway, so the question stays as the player wrote it. But without the note, "¿Y cuántos puntos da?" after the cost of moving a courtier up was asked back in 4 of 6 runs ("¿a qué te refieres?"), even with the conversation in front of it. With the note, it was answered in 4 of 4.
- **First turn:** there is nothing to rewrite, so no Gemini call is spent.

With these choices the agent understood the follow-ups of the test conversations ("lo" was the courtier, "las dos cosas" entering the castle and going up, "me paso" the iron), and no answer mixed in an earlier topic, even after two changes of topic. The failures that remained are listed in section 3.1.

In [13]:
# Constants
REWRITE_PROMPT = """You prepare the last question of a conversation about the rules of the board game White Castle for a search in the rulebook. Classify it into one of two kinds and return the question to search, in Spanish:
- "same_topic": it continues the topic of the earlier turns. If it depends on them (for example "y si...", "en ese caso", "eso", "lo", "las dos cosas", "arriba del todo", "sí", "siguiente"), rewrite it so that it can be understood on its own, replacing the references with what they refer to. If it can already be understood on its own, return it unchanged. If the player only confirms or asks to go on, write a question about the step the assistant offered to explain next, named by its content, not by its number.
- "new_topic": it can be understood on its own and is about a different topic. Return it unchanged, without adding anything from the earlier turns.
When you rewrite, use the words the assistant used in the conversation (they come from the rulebook) instead of the player's informal ones: for example "la sala del Daimio" instead of "arriba del todo". The rulebook calls points "puntos de clan": use that name when the player asks about points."""

TOPIC_CHANGE_NOTE = "Note: the player has changed topic. The earlier turns are about other rules: answer this question on its own."
FOLLOW_UP_NOTE = "Note: this question continues the conversation. On its own, it means: {question}"

### 2.3 The graph

The state holds the conversation (`messages`), the kind of the last question, the question to search and the chunks retrieved for it. The graph is compiled with the checkpointer, so every turn starts from the saved state of its conversation.
- **Pages in the context:** each chunk reaches Gemini under a header with its page (`[Página 5]`), followed by its section and its text, so the answer can cite the page.
- **Notes before the question:** `TOPIC_CHANGE_NOTE` for a new topic, or `FOLLOW_UP_NOTE` with the rewritten question for a follow-up (section 2.2).
- **Structured answer of the rewrite:** Gemini returns a small dictionary (kind and question) instead of free text, so the code can read the kind without parsing sentences.
- **`chat(question, thread_id)`:** sends one turn to the conversation `thread_id`; a new id starts a conversation from zero. Under each answer it shows the kind of question, the rewritten question when there is one, and the pages of the retrieved chunks. That shows the agent searches the rulebook on every question, and tells a failed search from a failed answer.
- **The graph is not drawn by LangGraph:** it draws it by sending the graph to an external web service (mermaid.ink). For three nodes in a row, the diagram of section 2 is enough.

In [14]:
class AgentState(TypedDict):
    """
    State that travels through the graph:
        - messages: the conversation, user questions and agent answers
        - question_kind: first_turn, same_topic or new_topic
        - search_query: the last question, as it is searched in the rulebook
        - chunks: the rulebook chunks retrieved for the last question
    """
    messages: Annotated[list, add_messages]
    question_kind: str
    search_query: str
    chunks: list


class Rewrite(TypedDict):
    """Structured answer of the rewrite: kind of the last question and the question to search."""
    kind: Literal["same_topic", "new_topic"]
    question: str


def build_agent():
    """Graph START → rewrite → retrieve → generate → END, compiled with an in-memory checkpointer."""
    graph = StateGraph(AgentState)
    graph.add_node("rewrite", rewrite)
    graph.add_node("retrieve", retrieve)
    graph.add_node("generate", generate)
    graph.add_edge(START, "rewrite")
    graph.add_edge("rewrite", "retrieve")
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", END)
    return graph.compile(checkpointer=InMemorySaver())


def rewrite(state):
    """Rewrite node: earlier turns + last question → kind + question to search (first turn: unchanged, no Gemini call)."""
    *history, question = state["messages"]
    if not history:
        return {"question_kind": "first_turn", "search_query": question.content}
    prompt = [SystemMessage(REWRITE_PROMPT), HumanMessage(build_rewrite_message(history, question.content))]
    result = rewriter.invoke(prompt)
    return {"question_kind": result["kind"], "search_query": result["question"]}


def build_rewrite_message(history, question):
    """Earlier turns as 'Jugador:' / 'Asistente:' lines + the last question."""
    speakers = {"human": "Jugador", "ai": "Asistente"}
    turns = "\n\n".join(f"{speakers[message.type]}: {message.text}" for message in history)
    return f"Conversation:\n\n{turns}\n\nLast question: {question}"


def retrieve(state):
    """Retrieve node: question to search → top RETRIEVAL_K chunks of the rulebook."""
    return {"chunks": vector_store.similarity_search(state["search_query"], k=RETRIEVAL_K)}


def generate(state):
    """Generate node: system prompt + earlier turns + note, chunks and question → Gemini answer."""
    *history, question = state["messages"]
    note = build_note(state["question_kind"], question.content, state["search_query"])
    user_message = f"{note}{build_user_message(question.content, state['chunks'])}"
    prompt = [SystemMessage(SYSTEM_PROMPT), *history, HumanMessage(user_message)]
    return {"messages": [llm.invoke(prompt)]}


def build_note(question_kind, question, search_query):
    """
    Note before the excerpts, empty if none:
        - new_topic → TOPIC_CHANGE_NOTE
        - same_topic, rewritten → FOLLOW_UP_NOTE with the rewritten question
    """
    if question_kind == "new_topic":
        return f"{TOPIC_CHANGE_NOTE}\n\n"
    if question_kind == "same_topic" and search_query != question:
        return f"{FOLLOW_UP_NOTE.format(question=search_query)}\n\n"
    return ""


def build_user_message(question, chunks):
    """Question + its chunks, each one under a header with its page."""
    excerpts = "\n\n".join(f"[Página {chunk.metadata['page']}]\n{chunk.page_content}" for chunk in chunks)
    return f"Rulebook excerpts:\n\n{excerpts}\n\nQuestion: {question}"


def chat(question, thread_id):
    """
    One turn of the conversation thread_id → answer shown as Markdown:
        - Below it, how the question was searched and the pages of the retrieved chunks
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke({"messages": [HumanMessage(question)]}, config)
    pages = ", ".join(f"p{chunk.metadata['page']}" for chunk in result["chunks"])
    display(Markdown(f"**{question}**\n\n{result['messages'][-1].text}\n\n*{describe_search(question, result)} · Retrieved: {pages}*\n\n---"))


def describe_search(question, result):
    """
    How the question was searched: its kind, plus the rewritten question if it changed
        Example: 'same_topic → ¿Cómo sube un cortesano de piso?'
    """
    if result["search_query"] != question:
        return f"{result['question_kind']} → {result['search_query']}"
    return result["question_kind"]


rewriter = llm.with_structured_output(Rewrite)
agent = build_agent()

## 3. Examples and chat

The finished agent: example questions that go through every kind of question of the system prompt, with their answers checked against the rulebook, and a chat cell to talk to it.

### 3.1 Examples

Two conversations. The first goes through the five kinds of question of the system prompt (section 2.1) and shows the memory with a follow-up question; the second asks that same follow-up with no conversation behind it.

| Conversation | Question | Kind | What it shows |
|---|---|---|---|
| `examples` | "¿Cuánto cuesta subir un cortesano dos pisos?" | Rule in the rulebook | A direct rule, answered with the cost, not with "Sí" |
| `examples` | "¿Y cuántos puntos da?" | Rule in the rulebook, follow-up | Memory: "da" refers to the courtier of the turn before. The points depend on the floor it starts from |
| `examples` | "¿Puedo pagar un dado con las monedas que me da la propia acción?" | Rule in the rulebook, deduced | The rulebook never says it literally: it follows from the order of the steps |
| `examples` | "¿Hay un tiempo máximo para cada turno?" | Not in the rulebook | "No encuentro esta regla en el manual" and a house rule, with no page |
| `examples` | "¿Qué me recomiendas para cenar esta noche?" | Not about White Castle | The fixed reply |
| `examples` | "Enséñame a jugar" | Learning to play | Only the list of steps, waiting for the player to start |
| `new_conversation` | "¿Y cuántos puntos da?" | Cannot be understood on its own | Without the conversation, the agent asks what it refers to instead of guessing |

In [15]:
# Constants
# Example conversations: thread_id → questions, in order
EXAMPLE_CONVERSATIONS = {
    "examples": [
        "¿Cuánto cuesta subir un cortesano dos pisos?",
        "¿Y cuántos puntos da?",                                            # follow-up: the courtier going up
        "¿Puedo pagar un dado con las monedas que me da la propia acción?",
        "¿Hay un tiempo máximo para cada turno?",                           # not in the rulebook
        "¿Qué me recomiendas para cenar esta noche?",                       # not about White Castle
        "Enséñame a jugar",                                                 # learning to play
    ],
    "new_conversation": [
        "¿Y cuántos puntos da?",                                            # the same follow-up, with no conversation
    ],
}

In [16]:
for thread_id, questions in EXAMPLE_CONVERSATIONS.items():
    display(Markdown(f"#### Conversation `{thread_id}`"))
    for question in questions:
        chat(question, thread_id)

#### Conversation `examples`

**¿Cuánto cuesta subir un cortesano dos pisos?**

5 nácares.

Para hacer subir a un cortesano dos pisos en el castillo mediante la acción de ascensión social, debes pagar un coste de 5 nácares (o de 2 nácares si decides subirlo solo un piso).

Lo tienes en la página 10.

*first_turn · Retrieved: p10, p11, p10, p10, p11*

---

**¿Y cuántos puntos da?**

Subir un cortesano dos pisos le permite alcanzar el segundo piso, lo que le otorgará 6 puntos de clan al final de la partida.

Durante el recuento final, cada cortesano otorga puntos según el lugar donde se encuentre: 1 punto si está en la puerta, 3 puntos en el primer piso, 6 puntos en el segundo piso y 10 puntos en el tercer piso (Daimio).

Lo tienes en la página 14.

*same_topic → ¿Cuántos puntos de clan da subir un cortesano dos pisos? · Retrieved: p14, p9, p15, p10, p10*

---

**¿Puedo pagar un dado con las monedas que me da la propia acción?**

No.

Para colocar un dado en una casilla cuyo valor es superior al del dado, debes pagar la diferencia en monedas a la reserva común en el momento de colocarlo y antes de realizar cualquier acción de esa casilla; por lo tanto, no puedes usar las monedas que consigas como recompensa posterior para pagar ese coste.

Lo tienes en la página 5.

*same_topic · Retrieved: p5, p17, p6, p5, p8*

---

**¿Hay un tiempo máximo para cada turno?**

No encuentro esta regla en el manual, ya que el reglamento no especifica un tiempo máximo de duración para los turnos de los jugadores.

Como sugerencia de regla casera para vuestra partida, podéis establecer un límite de tiempo de dos minutos por turno para mantener un ritmo ágil de juego.

*new_topic · Retrieved: p4, p13, p13, p12, p13*

---

**¿Qué me recomiendas para cenar esta noche?**

Solo puedo ayudarte con las reglas de White Castle.

*new_topic · Retrieved: p20, p19, p2, p1, p13*

---

**Enséñame a jugar**

1. Preparación del castillo.
2. Preparación de los jugadores.
3. Desarrollo de la partida.
4. Fin de ronda y fin de partida.

¿Estás preparado para el primer paso?

*new_topic · Retrieved: p3, p2, p3, p4, p2*

---

#### Conversation `new_conversation`

**¿Y cuántos puntos da?**

¿A qué regla o parte del juego te refieres con "eso"?

*first_turn · Retrieved: p14, p14, p14, p14, p15*

---

**What the examples show**
- **Each kind gets its format** (section 2.1): short answer, explanation and page for the rules; "No encuentro esta regla en el manual" with a labelled house rule and no page; the fixed reply for dinner; only the list of steps for "Enséñame a jugar".
- **Memory:** "¿Y cuántos puntos da?" is rewritten as "¿Cuántos puntos de clan da subir un cortesano dos pisos?", the scoring of page 14 comes first, and the answer is about the courtier of the turn before. It is only partly complete: it assumes the courtier starts at the castle gate (second floor, 6 points) and does not say that from the first floor it would reach the Daimio's hall (10 points), although it lists the points of every floor.
- **The same question with no conversation** (`new_conversation`) is not understood, as expected: the agent asks what it refers to instead of guessing.
- **Deduced rule:** right. The difference is paid when the die is placed, before the action, so the coins of the action cannot pay for it (page 5). In an earlier run this question was answered "Sí": `gemini-3.5-flash-lite` ignores `temperature`, so answers can change between runs (section 0 explains why a larger model is not used).

**Limits seen in the development tests** (also in the README):
- **Answers vary between runs**, mostly in questions that must be deduced from several rules. In the wrong answers seen, the right pages had been retrieved: the error was in the reasoning, not in the search.
- **About one follow-up in five was rewritten without a key word** (for example, "¿Y qué gano si llega arriba del todo?" lost the courtier), and the search then missed its page.
- **A rule split across two chunks:** the exception of the gardeners in the last round is in a chunk that is not retrieved with the question about them, so "¿Y en la última ronda también?" got a wrong "Sí".
- **Learning to play:** in one run, the third turn searched the step already explained instead of the next one.

### 3.2 Chat

Run the cell and type the questions in the box that opens (in VS Code, at the top of the window). Each answer shows, below it, how the question was searched and the pages of the retrieved chunks:
- **`nueva`:** starts a new conversation, with no memory of the earlier one
- **`salir`** or an empty line: ends the chat

In [ ]:
def chat_loop():
    """
    Chat with the agent from the keyboard, one conversation at a time:
        - 'nueva' → new conversation
        - 'salir' or an empty line → end
    """
    conversation = 1
    print("Escribe tu pregunta. 'nueva' empieza otra conversación; 'salir' termina.")
    while True:
        question = input("Pregunta: ").strip()
        if question.lower() in ("", "salir"):
            print("Fin del chat.")
            return
        if question.lower() == "nueva":
            conversation += 1
            print("Nueva conversación.")
        else:
            chat(question, f"chat_{conversation}")


chat_loop()